<a href="https://colab.research.google.com/github/TanyaSrivastava444/marketing-mix-model-Staffing/blob/main/Notebooks/Adstock_Transformation_%26_Saturation_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd

In [17]:
df = pd.read_pickle('/content/df_csv.pkl')

In [18]:
df.head()

,job_board_spend,linkedin_spend,events_spend,youtube_ads_spend,google_ads_spend,client_outreach_spend,unemployment_rate,job_openings,payrolls,placements
week,,,,,,,,,,
2021-01-03,-0.524041,-1.891748,-1.433125,-1.400482,-1.917682,0.731560,2.072289,-2.736924,-1.917194,57.604871
2021-01-10,-0.751895,-0.725998,-0.598354,-0.168823,-0.335504,0.291253,2.072289,-2.736924,-1.917194,81.520929
2021-01-17,-0.188687,-1.325883,-3.238207,-1.529308,-0.998541,-0.652469,2.072289,-2.736924,-1.917194,90.073056
2021-01-24,0.421053,-0.218631,-1.249438,1.122995,-0.064334,-1.182664,2.072289,-2.736924,-1.917194,101.153439
2021-01-31,-0.442846,-0.994995,-0.503811,0.694020,-0.819811,1.653343,2.072289,-2.736924,-1.917194,105.179272


In [56]:
# normalize the target
y = (y - y.mean()) / y.std()

# Calculation of Decay by grid search



In [19]:
# Adstock Definition

import numpy as np
def adstock(x, decay):

    result = np.zeros(len(x))
    result[0] = x[0]

    for t in range(1, len(x)):
        result[t] = x[t] + decay * result[t-1]

    return result


In [20]:
# define decay grid
decay_grid = np.arange(0.1, 0.9, 0.05)

In [21]:
# define predictor and target. y=Xβ+ϵ

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [23]:



y = df["placements"].values

In [24]:
channels = [
"job_board_spend",
"linkedin_spend",
"events_spend",
"youtube_ads_spend",
"google_ads_spend",
"client_outreach_spend"
]

decay_results = {}

for channel in channels:

    spend = df[channel].values

    best_decay = None
    best_score = -np.inf

    for decay in decay_grid:

        adstocked = adstock(spend, decay)

        X = adstocked.reshape(-1,1)

        model = LinearRegression().fit(X,y)

        score = model.score(X,y)

        if score > best_score:

            best_score = score
            best_decay = decay

    decay_results[channel] = best_decay

In [28]:
# O/p optimal decay
for ch,dec in decay_results.items():
    print(f"{ch}: {dec}")

job_board_spend: 0.6000000000000002
linkedin_spend: 0.5500000000000002
events_spend: 0.8000000000000002
youtube_ads_spend: 0.7000000000000002
google_ads_spend: 0.6500000000000001
client_outreach_spend: 0.7000000000000002


In [29]:
decay_results

{'job_board_spend': np.float64(0.6000000000000002),
 'linkedin_spend': np.float64(0.5500000000000002),
 'events_spend': np.float64(0.8000000000000002),
 'youtube_ads_spend': np.float64(0.7000000000000002),
 'google_ads_spend': np.float64(0.6500000000000001),
 'client_outreach_spend': np.float64(0.7000000000000002)}

### For production we can't use separate decay rates for different channels for an MMM modelling so we will have to generate a unified decay value for all channels


In [30]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import itertools

In [32]:
# This generates all possible combinations of decays across channels.
itertools.product()

In [33]:
# generate decay combinations
# decay_grid = [0.1,0.3,0.5,0.7] and channels = 6 then 4^6 = 4096

decay_combinations = list(itertools.product(decay_grid, repeat=len(channels)))

In [35]:
# best model storage
best_score = -np.inf
best_decays = None
best_model = None

In [38]:
# Grid Search Loop

for decays in decay_combinations:
  X = pd.DataFrame(index=df.index)

for i,channel in enumerate(channels):

    decay = decays[i]

    adstocked = adstock(df[channel].values, decay)

    X[channel+"_adstock"] = adstocked

    X = pd.DataFrame(index=df.index)

for i,channel in enumerate(channels):

    decay = decays[i]

    adstocked = adstock(df[channel].values, decay)

    X[channel+"_adstock"] = adstocked
    y = df["placements"].values

model = LinearRegression().fit(X,y)



KeyboardInterrupt: 

In [ ]:
# model Evaluate
y_pred = model.predict(X)

score = r2_score(y,y_pred)

In [ ]:
# store Best Decay Combination

if score > best_score:

    best_score = score
    best_decays = decays
    best_model = model

In [ ]:
# final decay values

optimal_decays = dict(zip(channels,best_decays))

print("Optimal Decays:")
print(optimal_decays)

print("Best Model R2:",best_score)

In [ ]:
### Create Final Adstock Features For MMM'

for ch in channels:

    decay = optimal_decays[ch]

    df[ch+"_adstock"] = adstock(df[ch].values, decay)

# Decision on Decay

 So estimation of decay can be done by grid search , or refer to MMM standards  but since the decay rate for some channels are being computed very high then we need to take a step back and use conservative half life decay values to avoid over contributon by a channel and to make the interpretation easier by the stakeholders.
So we will use : Half-life = time required for marketing impact to decay by 50%.
i.e. decay = 0.4
Use 0.40 across the board.
It's the empirical median across digital channels in services industries.It won't catastrophically overstate carryover, and it gives the data room to pull the decay up or down when you move to Phase 2.

Business teams understand time duration better than decay coefficients.



# Adstock Transformation

carryover effect
 = Adstockt​=Spendt​+λAdstockt−1​    where lambda = 0.4 decay

In [39]:
def adstock(x, decay=0.4):
    result = np.zeros(len(x))
    result[0] = x[0]

    for t in range(1, len(x)):
        result[t] = x[t] + decay * result[t-1]

    return result

In [40]:
for ch in channels:
    df[ch + "_adstock"] = adstock(df[ch], decay=0.5)

/tmp/ipykernel_148/3174554621.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  result[0] = x[0]
/tmp/ipykernel_148/3174554621.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  result[t] = x[t] + decay * result[t-1]
/tmp/ipykernel_148/3174554621.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  result[0] = x[0]
/tmp/ipykernel_148/3174554621.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a 

In [ ]:
# 3. Saturation Modeling (Diminishing Returns)
### Marketing Reality
### Spend increases do not produce linear results.   after a certain time the spend response curve plateaus due to ad fatigue and the effect of ads/campaigns saturation. the first few days of a ad scooped up most inetersted customers and then it dies down
hill function is used


In [41]:
def hill_function(x, alpha=1.5, gamma=1):
    return (x**alpha) / (gamma**alpha + x**alpha)

In [42]:
adstock_cols = [c+"_adstock" for c in channels]

for col in adstock_cols:
    df[col+"_sat"] = hill_function(df[col])

# Seasonality Modeling (Fourier Terms)
sin(2πkt/T)


 ;    cos(2πkt/T)

In [43]:
import numpy as np

def fourier_series(df, period, order):
    t = np.arange(len(df))

    for i in range(1, order+1):
        df[f'sin_{i}'] = np.sin(2 * np.pi * i * t / period)
        df[f'cos_{i}'] = np.cos(2 * np.pi * i * t / period)

    return df

df = fourier_series(df, period=52, order=2)

In [45]:

macro_vars = [
"unemployment_rate",
"job_openings",
"payrolls"
]

# Bayesian Regression (PyMC)

In [60]:
from sklearn.preprocessing import StandardScaler
import pymc as pm
import numpy as np

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

y_scaled = (y - y.mean()) / y.std()

with pm.Model() as mmm_model:

    intercept = pm.Normal("intercept", 0, 1)

    betas = pm.HalfNormal("betas", sigma=1, shape=X_scaled.shape[1])

    sigma = pm.HalfNormal("sigma", 1)

    mu = intercept + pm.math.dot(X_scaled, betas)

    placements = pm.Normal("placements", mu=mu, sigma=sigma, observed=y_scaled)

    trace = pm.sample(2000, tune=1000, target_accept=0.9)

SamplingError: Initial evaluation of model at starting point failed!
Starting values:
{'intercept': array(0.5407024), 'betas_log__': array([ 0.47972917, -0.51792782, -0.17580941,  0.5672934 ,  0.8741314 ,
       -0.28811051, -0.49390324, -0.53999237,  0.54253678,  0.54028069,
       -0.52833691,  0.86888667, -0.33580675]), 'sigma_log__': array(0.56295374)}

Logp initial evaluation results:
{'intercept': np.float64(-1.07), 'betas': np.float64(-15.07), 'sigma': np.float64(-1.2), 'placements': np.float64(nan)}
You can call `model.debug()` for more details.

In [54]:
X = np.nan_to_num(X)
y = np.nan_to_num(y)

In [51]:
## Channel Contribution Estimation

beta_means = trace.posterior['betas'].mean(dim=("chain","draw")).values

contribution = X * beta_means

channel_contribution = pd.DataFrame(
contribution,
columns=features,
index=df.index
)

NameError: name 'trace' is not defined